<a href="https://colab.research.google.com/github/oliversebastianmartinesdiaz-cmyk/Applied-Artificial-Intelligence-Course-with-Llama/blob/main/E2_Challenge_2_ON_Conversational_Agent_Design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: DISEÑO DE AGENTES CONVERSACIONALES**

Una vez vista la masterclass ***Diseño de Agentes Conversacionales***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.


De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/18wkM2LSfY39VZ2GBdcPCY2Qw0EzewJsx?usp=drive_link).

## **LÓGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**


#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [1]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

VERIFY_TOKEN = "mi_token"
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')

print("Credenciales cargadas correctamente.")

Credenciales cargadas correctamente.


#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [2]:
# Instalar Ollama

!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 64 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (400 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/loca

In [3]:
# Iniciar el servidor de Ollama en segundo plano

import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Ollama listo.")


Ollama listo.


#### **INSTALAR Y LEVANTAR LLAMA STACK**

In [4]:
# Instalar dependencias

!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.3/375.3 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.2/569.2 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 6.6 MB/s eta 0:00:00
  

In [5]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado

import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

# Generar un config de Llama Stack con el modelo de Ollama ya registrado
config_path = os.path.join(os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Levantar el servidor con ese config
proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321")
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [6]:
# Configurar el servidor del webhook

from fastapi import FastAPI, Request, BackgroundTasks
import requests
import ollama

app = FastAPI()

mensajes_recibidos = []
usuarios_conocidos = set()

In [8]:
# Definir función de normalización de número

def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Se define una estructura en memoria que guarda, por cada número de teléfono, el historial reciente de mensajes — sin esto, Llama trataría cada mensaje como si fuera el primero de la conversación. Esta es una implementación manual y simplificada del concepto; la lectura del Tema menciona **Llama Stack** como el framework que estandariza esta pieza (junto con herramientas y seguridad) para no reconstruirla en cada proyecto. **Nota:** el patrón exacto de integración de Llama Stack debe confirmarse contra su documentación vigente antes de grabar o dar la sesión en vivo.

In [9]:
# Una conversacion de Llama Stack por numero de telefono

conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]

#### **GENERAR LA RESPUESTA CON HISTORIAL Y UNA REGLA FIJA**

Se define la lógica que arma cada respuesta: si el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento (una decisión que conviene fijar por regla, no dejarla al modelo); en cualquier otro caso, se reenvía todo el historial reciente a Llama, para que pueda resolver referencias a mensajes anteriores.

In [10]:
# Definir la funcion de generacion de respuesta con memoria

FRASES_ESCALAMIENTO = ["hablar con alguien", "hablar con una persona", "agente humano"]

SYSTEM_PROMPT = (
    "Eres el asistente de agendado de citas de una clinica dental. "
    "Responde de forma breve (2-3 lineas)."
)

def generar_respuesta(numero, texto_usuario):
    texto_low = texto_usuario.lower()
    if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
        return "Claro, en un momento un miembro del equipo te contacta directamente."

    conversation_id = obtener_conversacion(numero)
    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT,
    )
    return respuesta.output_text

## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [11]:
# Definir endpoint de verificacion de credenciales

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params

    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))

    return {"error": "token invalido"}

In [12]:
# Definir endpoint de recepcion de mensajes

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass

    return {"status": "ok"}

In [13]:
# Definir funcion de envio de respuesta

respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v25.0/{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)
    respuestas_enviadas.append({"numero": numero, "status": r.status_code, "texto": r.text})

In [14]:
# Definir funcion de procesamiento: regla fija de escalamiento o respuesta de Llama
import threading

locks_por_numero = {}  # { numero: threading.Lock() }

def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]

errores_procesamiento = []

def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            texto_low = texto.lower()
            #if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                #enviar_respuesta(numero, "Claro, en un momento un miembro del equipo te contacta directamente.")
            #else:
            respuesta = generar_respuesta(numero, texto)
            enviar_respuesta(numero, respuesta)
        except Exception as e:
            errores_procesamiento.append({"numero": numero, "texto": texto, "error": repr(e)})


### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [15]:
# Instalar e iniciar tunel publico con ngrok

!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel_publico = ngrok.connect(8000)
print("URL a configurar como Callback URL en el dashboard de Meta:")
print(tunel_publico.public_url + "/webhook")

def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()

URL a configurar como Callback URL en el dashboard de Meta:
https://superjet-jimmy-consensus.ngrok-free.dev/webhook


### **PROBAR EL FLUJO COMPLETO**

Se manda un primer mensaje mencionando el servicio dental, y un segundo mensaje que depende del primero — para confirmar que el historial sí se está reenviando.

In [34]:
# Mandar el primer mensaje, mencionando el servicio

payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "529711199885", "text": {"body": "Quiero una cita dental"}} # 5215500000000
    ]}}]}]
}

r1 = requests.post("http://127.0.0.1:8000/webhook", json=payload_1)
print("Mensaje 1:", r1.status_code, r1.text)

INFO:     127.0.0.1:57382 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 1: 200 {"status":"ok"}


In [36]:
# Mandar un segundo mensaje que depende del primero, sin repetir el servicio

payload_2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "529711199885", "text": {"body": "¿Tienen horario el sábado?"}}
    ]}}]}]
}

r2 = requests.post("http://127.0.0.1:8000/webhook", json=payload_2)
print("Mensaje 2:", r2.status_code, r2.text)

# la respuesta demora aprox. de 1 a 2 min. al ejecutar local en colab

INFO:     127.0.0.1:49792 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 2: 200 {"status":"ok"}


In [37]:
# Revisar que paso con los mensajes
print('Mensajes recibidos:', len(mensajes_recibidos))
print('Respuestas enviadas', respuestas_enviadas)
print('Errores de procesamiento', errores_procesamiento)
print("Conversaciones: ", conversaciones_por_numero)
print("Llama stack vivo: ", proceso_llama_stack.poll() is None )
print("Servidor vivo:", hilo.is_alive())

Mensajes recibidos: 7
Respuestas enviadas [{'numero': '529711199885', 'status': 403, 'texto': '{"error":{"message":"(#131005) Access denied","code":131005,"type":"OAuthException","error_data":{"messaging_product":"whatsapp","details":"There was a problem with the access token or permissions you are using for the API call."},"fbtrace_id":"ASFvpGX0xOfjygOUbEqiXP9"}}'}, {'numero': '529711199885', 'status': 403, 'texto': '{"error":{"message":"(#131005) Access denied","code":131005,"type":"OAuthException","error_data":{"messaging_product":"whatsapp","details":"There was a problem with the access token or permissions you are using for the API call."},"fbtrace_id":"APFl5DsQjquHeqp2_Vp1WnF"}}'}, {'numero': '529711199885', 'status': 403, 'texto': '{"error":{"message":"(#131005) Access denied","code":131005,"type":"OAuthException","error_data":{"messaging_product":"whatsapp","details":"There was a problem with the access token or permissions you are using for the API call."},"fbtrace_id":"AOYLgs

In [ ]:
# Ver mensajes recibidos

mensajes_recibidos

In [ ]:
# Verificar almacenamiento en Llama Stack

numero = normalizar_numero_mx("525626131742")  # tu numero real de WhatsApp
conv_id = conversaciones_por_numero[numero]

items = client.conversations.items.list(conversation_id=conv_id)
for item in items.data:
    print(item.role, "->", item.content)

In [ ]:
# Finalizar proceso de tunelización

ngrok.kill()
print('Proceso de ngrok cerrado.')

INFO:     Started server process [399]


## **CHALLENGE: AGENTE DE UN ESTUDIO DE FOTOS CON DRONES**


Una vez visto el ***Hands-On: Diseño de Agentes Conversacionales***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Prepara el entorno:** Instala y arranca Ollama en Colab, descarga el modelo Llama 3.1 8B, instala y levanta **Llama Stack** con Ollama registrado, y carga las credenciales de WhatsApp desde **Colab Secrets**.

In [1]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

VERIFY_TOKEN = "mi_token"
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')

print("Credenciales cargadas correctamente.")

Credenciales cargadas correctamente.


In [2]:
# Instalar Ollama (requiere zstd, que Colab no trae por defecto)
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
E: dpkg was interrupted, you must manually run 'dpkg --configure -a' to correct the problem. 
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
# Iniciar el servidor de Ollama en segundo plano
import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Ollama listo.")



Ollama listo.


In [4]:
# Instalar dependencias
!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 16.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.3/375.3 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.2/569.2 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 4.7 MB/s eta 0:00:00
  

In [5]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado
import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

config_path = os.path.join(os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321")
    print("Llama Stack listo.")


Proceso vivo: True
Llama Stack listo.


In [6]:
# Leer credenciales desde Colab Secrets
from google.colab import userdata

VERIFY_TOKEN = "mi_token"
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')

print("Credenciales cargadas correctamente.")


Credenciales cargadas correctamente.


**2. Construye la memoria de conversación:** Crea una conversación de **Llama Stack** por número de teléfono (`client.conversations.create()`) con una memoria de respuesta.

In [7]:
# Conversacion de Llama Stack por numero de telefono
conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]


In [8]:
# Definir la funcion de generacion de respuesta con memoria y limite de historial
FRASES_ESCALAMIENTO = ["hablar con alguien", "hablar con una persona", "agente humano", "humano", "asesor", "fotografo"]
FRASES_URGENCIA = ["urgencia", "emergencia", "hoy mismo", "inmediato", "ahora mismo", "clima", "lluvia", "viento fuerte"]

SYSTEM_PROMPT = (
    "Eres el asistente virtual de 'AeroFoto Studios', estudio profesional de fotografia y video aereo con drones. "
    "Tu mision es orientar amablemente sobre paquetes para bodas, eventos, bienes raices e inspecciones, "
    "y agendar citas para sesiones fotograficas o levantamientos aereos. "
    "Responde de forma concisa, profesional y breve (maximo 2 a 3 lineas)."
)

LIMITE_HISTORIAL = 4  # Maximo 4 mensajes retenidos en el historial de la conversacion

def recortar_historial(conv_id, limite=LIMITE_HISTORIAL):
    try:
        items = client.conversations.items.list(conversation_id=conv_id)
        if len(items.data) > limite:
            exceso = len(items.data) - limite
            for item in items.data[:exceso]:
                try:
                    client.conversations.items.delete(conversation_id=conv_id, item_id=item.id)
                except Exception:
                    pass
    except Exception:
        pass

def generar_respuesta(numero, texto_usuario):
    texto_low = texto_usuario.lower()

    # Regla 1: Urgencia o alerta de vuelo (evento express o condiciones de clima)
    if any(frase in texto_low for frase in FRASES_URGENCIA):
        return (
            "¡ATENCION! Para coberturas aereas de urgencia hoy mismo o dudas de vuelo por clima, "
            "por favor comunicate de inmediato a nuestra linea prioritaria de pilotos al 55-911-DRON."
        )

    # Regla 2: Escalamiento a humano
    if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
        return "Claro, en un momento un piloto fotografo o asesor de AeroFoto Studios te contacta directamente."

    conversation_id = obtener_conversacion(numero)
    recortar_historial(conversation_id, LIMITE_HISTORIAL)

    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT,
    )

    recortar_historial(conversation_id, LIMITE_HISTORIAL)
    return respuesta.output_text


**3. Agrega la regla fija y protege el procesamiento por número:**

   * Cuando el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento.
   * Igual que en el Hands-On, usa un lock por número de teléfono para que dos mensajes seguidos del mismo usuario no disparen dos llamadas en paralelo sobre la misma conversación.

In [9]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario
import threading

locks_por_numero = {}

def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]


In [10]:
# Definir funcion de procesamiento: regla fija de escalamiento o respuesta de Llama
errores_procesamiento = []

def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            texto_low = texto.lower()
            if any(frase in texto_low for frase in FRASES_URGENCIA):
                enviar_respuesta(
                    numero,
                    "¡ATENCION! Para coberturas aereas de urgencia hoy mismo o dudas de vuelo por clima, comunicate de inmediato a nuestra linea prioritaria al 55-911-DRON."
                )
            elif any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                enviar_respuesta(
                    numero,
                    "Claro, en un momento un piloto fotografo o asesor de AeroFoto Studios te contacta directamente."
                )
            else:
                respuesta = generar_respuesta(numero, texto)
                enviar_respuesta(numero, respuesta)
        except Exception as e:
            errores_procesamiento.append({"numero": numero, "texto": texto, "error": repr(e)})


**4. Define los endpoints del webhook:** usa `BackgroundTasks` para responder de inmediato a Meta, delegando el procesamiento real a la función anterior.

In [11]:
# Configurar el servidor del webhook
from fastapi import FastAPI, Request, BackgroundTasks
import requests

app = FastAPI()

mensajes_recibidos = []
usuarios_conocidos = set()


In [12]:
# Definir funcin de normalizacin de nmero
def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero


In [13]:
# Definir endpoint de verificacion de credenciales
@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params
    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))
    return {"error": "token invalido"}


In [14]:
# Definir endpoint de recepcion de mensajes
@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)
    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass
    return {"status": "ok"}


In [15]:
# Definir funcion de envio de respuesta
respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v25.0/{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }
    r = requests.post(url, headers=headers, json=payload)
    respuestas_enviadas.append({"numero": numero, "status": r.status_code, "texto": r.text})


**5. Expón el servidor con ngrok y prueba el flujo completo:**

   * Manda una conversación de al menos 4 mensajes y confirma que el historial se recorta correctamente.
   * Manda un mensaje con una palabra de urgencia y confirma que dispara la regla fija en vez de una respuesta de Llama.

In [16]:
# Instalar e iniciar tunel publico con ngrok
!pip install pyngrok --quiet
from pyngrok import ngrok
import uvicorn
import threading

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel_publico = ngrok.connect(8000)
print("URL a configurar como Callback URL en el dashboard de Meta:")
print(tunel_publico.public_url + "/webhook")

def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()


URL a configurar como Callback URL en el dashboard de Meta:
https://superjet-jimmy-consensus.ngrok-free.dev/webhook


In [20]:
# Mandar el primer mensaje, mencionando el servicio
payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "529711199885", "text": {"body": "Hola, quiero agendar una sesion de fotos y video con drones para una boda"}}
    ]}}]}]
}
r1 = requests.post("http://127.0.0.1:8000/webhook", json=payload_1)
print("Mensaje 1:", r1.status_code, r1.text)


INFO:     127.0.0.1:40708 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 1: 200 {"status":"ok"}


In [ ]:
# Mandar un segundo mensaje que depende del primero, sin repetir el servicio
payload_2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "529711199885", "text": {"body": "¿Tienen disponibilidad este sabado por la tarde?"}}
    ]}}]}]
}
r2 = requests.post("http://127.0.0.1:8000/webhook", json=payload_2)
print("Mensaje 2 (prueba de memoria):", r2.status_code, r2.text)

# Mensajes 3 y 4 (para verificar que se superan 4 mensajes y el historial se recorta correctamente)
payload_3 = {"entry": [{"changes": [{"value": {"messages": [{"from": "5215626131742", "text": {"body": "El evento sera en un jardin abierto en Cuernavaca"}}]}}]}]}
requests.post("http://127.0.0.1:8000/webhook", json=payload_3)

payload_4 = {"entry": [{"changes": [{"value": {"messages": [{"from": "5215626131742", "text": {"body": "Â¿Cual es el costo del paquete que incluye video 4K y fotos editadas?"}}]}}]}]}
requests.post("http://127.0.0.1:8000/webhook", json=payload_4)

# Mensaje 5: Prueba de REGLA FIJA DE URGENCIA (dispara respuesta fija inmediata sin Llama)
payload_urgencia = {"entry": [{"changes": [{"value": {"messages": [{"from": "5215626131742", "text": {"body": "¡Es una urgencia, tenemos un evento hoy mismo y necesitamos piloto de dron ya!"}}]}}]}]}
r_urg = requests.post("http://127.0.0.1:8000/webhook", json=payload_urgencia)
print("Mensaje de Urgencia enviado:", r_urg.status_code, r_urg.text)


In [ ]:
# Verificar respuestas
import time
time.sleep(10)  # Breve espera para que los background tasks procesen
respuestas_enviadas


In [ ]:
# Ver mensajes recibidos
mensajes_recibidos


In [ ]:
# Verificar almacenamiento en Llama Stack
numero = normalizar_numero_mx("529711199885")
if numero in conversaciones_por_numero:
    conv_id = conversaciones_por_numero[numero]
    items = client.conversations.items.list(conversation_id=conv_id)
    print(f"Total de mensajes en memoria: {len(items.data)} (Limite maximo: {LIMITE_HISTORIAL})")
    for item in items.data:
        print(item.role, "->", item.content)
else:
    print("No se encontró conversación registrada para el número.")


In [ ]:
# Finalizar proceso de tunelizacin
ngrok.kill()
print('Proceso de ngrok cerrado.')
